# 001 · Unit Testing with unittest

> **Standard Library › Testing & Debugging** · Python 3.14

`unittest` is Python's built-in, class-based testing framework. It was modelled on JUnit, so if you have used xUnit.net, NUnit or MSTest you already know its shape: test classes, test methods, assertions, setup/teardown and a runner.

**In this notebook you will learn:**

- how xUnit/NUnit/MSTest concepts map onto `unittest.TestCase`
- the assertion methods you will use 90% of the time, including `assertRaises` as a context manager
- per-test and per-class fixtures (`setUp`, `tearDown`, `setUpClass`, `addCleanup`)
- data-driven tests with `subTest` (the `[Theory]` / `[TestCase]` equivalent)
- skipping tests and marking expected failures
- how to run tests inside a notebook, and from the command line with `python -m unittest`
- why most real projects use **pytest** instead (and how it relates to `unittest`)

## .NET mental model

| xUnit.net | NUnit | MSTest | Python `unittest` |
|---|---|---|---|
| test class (any public class) | `[TestFixture]` | `[TestClass]` | subclass of `unittest.TestCase` |
| `[Fact]` | `[Test]` | `[TestMethod]` | any method whose name starts with `test` |
| `[Theory]` + `[InlineData]` | `[TestCase(...)]` | `[DataRow(...)]` | a loop with `with self.subTest(...)` |
| constructor / `IDisposable.Dispose` | `[SetUp]` / `[TearDown]` | `[TestInitialize]` / `[TestCleanup]` | `setUp()` / `tearDown()` |
| `IClassFixture<T>` | `[OneTimeSetUp]` / `[OneTimeTearDown]` | `[ClassInitialize]` / `[ClassCleanup]` | `@classmethod setUpClass` / `tearDownClass` |
| `Assert.Equal(expected, actual)` | `Assert.That(actual, Is.EqualTo(expected))` | `Assert.AreEqual(expected, actual)` | `self.assertEqual(first, second)` |
| `Assert.Throws<T>(() => ...)` | `Assert.Throws<T>(() => ...)` | `Assert.ThrowsException<T>(...)` | `with self.assertRaises(T): ...` |
| `[Fact(Skip = "...")]` | `[Ignore("...")]` | `[Ignore]` | `@unittest.skip("...")` |
| `dotnet test` | `dotnet test` | `dotnet test` | `python -m unittest` |

The big differences: there are no attributes — **naming conventions** (`test_*` methods, `test*.py` files) do the discovery — and assertions are **methods on `self`**, not static methods on an `Assert` class.

## Your first test case

Here is some code under test, and a test class for it. In C# with xUnit you would write:

```csharp
public class PriceTests
{
    [Fact]
    public void Discount_IsApplied() =>
        Assert.Equal(90m, Pricing.ApplyDiscount(100m, 10));

    [Fact]
    public void NegativePercent_Throws() =>
        Assert.Throws<ArgumentOutOfRangeException>(() => Pricing.ApplyDiscount(100m, -5));
}
```

In [ ]:
import unittest

def apply_discount(price: float, percent: float) -> float:
    if not 0 <= percent <= 100:
        raise ValueError(f"percent must be 0-100, got {percent}")
    return round(price * (1 - percent / 100), 2)


class ApplyDiscountTests(unittest.TestCase):
    def test_discount_is_applied(self):
        self.assertEqual(apply_discount(100, 10), 90)

    def test_zero_percent_returns_price(self):
        self.assertEqual(apply_discount(59.99, 0), 59.99)

    def test_negative_percent_raises(self):
        with self.assertRaises(ValueError):
            apply_discount(100, -5)

    def helper_not_a_test(self):    # no "test" prefix -> not collected
        raise RuntimeError("never runs")

print("defined", ApplyDiscountTests.__name__)

## Running tests inside a notebook

In a script you would end with `unittest.main()`, but that calls `sys.exit()` and parses `sys.argv` — both wrong inside Jupyter. The most predictable notebook pattern is to load **one specific `TestCase`** and run it with a `TextTestRunner` that writes to `sys.stdout`:

In [ ]:
import sys

loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(ApplyDiscountTests)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)

print(result.wasSuccessful(), result.testsRun)   # True 3

`unittest.main()` also works if you neutralise its two side effects: give it a fake `argv` (otherwise it tries to parse the kernel's own command-line arguments) and `exit=False`.

⚠️ **Gotcha:** `unittest.main()` discovers **every** `TestCase` subclass in the `__main__` module — in a notebook that means every test class you have defined in *any* earlier cell. That is why this notebook prefers the loader approach above. Its output also goes to `stderr` by default.

In [ ]:
program = unittest.main(argv=["ignored"], exit=False, verbosity=2)
print("success:", program.result.wasSuccessful())

For the failing-test demos below we want short output with no long tracebacks (which would also contain machine-specific file paths), so here is a small helper that runs a `TestCase` silently and prints just the essentials:

In [ ]:
import io

def run_quiet(*cases: type[unittest.TestCase]) -> None:
    suite = unittest.TestSuite(loader.loadTestsFromTestCase(c) for c in cases)
    result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    for test, tb in result.failures + result.errors:
        # the first un-indented line after "Traceback ..." is the exception message
        message = next(l for l in tb.splitlines() if l and not l.startswith((" ", "Traceback")))
        print(f"FAIL {test.id().removeprefix('__main__.')}: {message}")
    print(f"ran={result.testsRun} failures={len(result.failures)} errors={len(result.errors)} "
          f"skipped={len(result.skipped)} expected_failures={len(result.expectedFailures)}")

run_quiet(ApplyDiscountTests)

## Assertion methods

Every assertion accepts an optional final `msg=` argument that is added to the failure message (like the `message` parameter on MSTest's `Assert.AreEqual`).

| C# (xUnit) | Python `unittest` |
|---|---|
| `Assert.Equal(a, b)` / `Assert.NotEqual` | `assertEqual(a, b)` / `assertNotEqual` |
| `Assert.True(x)` / `Assert.False(x)` | `assertTrue(x)` / `assertFalse(x)` (truthiness!) |
| `Assert.Same(a, b)` | `assertIs(a, b)` / `assertIsNot` |
| `Assert.Null(x)` / `Assert.NotNull(x)` | `assertIsNone(x)` / `assertIsNotNone(x)` |
| `Assert.Contains(item, collection)` | `assertIn(item, container)` / `assertNotIn` |
| `Assert.IsType<T>(x)` / `IsAssignableFrom` | `assertIsInstance(x, T)` |
| `Assert.Equal(0.3, x, precision: 7)` | `assertAlmostEqual(x, 0.3)` (7 decimal places by default) |
| `Assert.InRange` | `assertGreater`, `assertLess`, `assertGreaterEqual`, `assertLessEqual` |
| `Assert.Equivalent` (order-insensitive) | `assertCountEqual(a, b)` (same elements, any order) |
| `Assert.Matches(regex, s)` | `assertRegex(s, regex)` |
| `Assert.Throws<T>` + `.Message` check | `assertRaises(T)` / `assertRaisesRegex(T, regex)` |

In [ ]:
class AssertionShowcase(unittest.TestCase):
    def test_common_assertions(self):
        self.assertTrue([1])                       # non-empty list is truthy
        self.assertFalse("")                       # empty string is falsy
        self.assertIn("bob", ["alice", "bob"])
        self.assertIsNone({}.get("missing"))
        self.assertIsInstance(3, int)
        self.assertAlmostEqual(0.1 + 0.2, 0.3)     # assertEqual would fail!
        self.assertCountEqual([3, 1, 2], [1, 2, 3])
        self.assertRegex("order-1042", r"^order-\d+$")
        self.assertGreater(10, 3)

run_quiet(AssertionShowcase)

⚠️ **Gotcha:** `assertTrue(x)` checks **truthiness**, not `x is True` — `assertTrue([0])` passes. Use `assertIs(x, True)` if you really mean the boolean. Also, prefer the specific assertion (`assertIn`, `assertIsNone`, …) over `assertTrue(a in b)`: the specific ones produce much better failure messages.

## Exceptions: `assertRaises` and `assertRaisesRegex`

`assertRaises` is used as a **context manager** — the code that should throw goes in the `with` block. Keep the context manager as `cm` to inspect the exception afterwards, just like capturing the return value of `Assert.Throws<T>` in xUnit:

```csharp
var ex = Assert.Throws<ArgumentException>(() => Parse("abc"));
Assert.Contains("abc", ex.Message);
```

In [ ]:
class ExceptionTests(unittest.TestCase):
    def test_inspect_exception(self):
        with self.assertRaises(ValueError) as cm:
            apply_discount(100, 150)
        self.assertIn("150", str(cm.exception))

    def test_message_matches_regex(self):
        with self.assertRaisesRegex(ValueError, r"0-100, got -\d+"):
            apply_discount(100, -1)

    def test_wrong_exception_type_is_an_error(self):
        with self.assertRaises(KeyError):         # a ValueError escapes instead
            apply_discount(100, 500)

run_quiet(ExceptionTests)

Note the distinction in the summary: a failed *assertion* counts as a **failure**, while any other exception escaping the test (here the unexpected `ValueError`) counts as an **error** — the same distinction NUnit's runner reports.

## What a failure looks like

`assertEqual` produces rich diffs for strings, lists, dicts and sets. The `msg=` argument is appended to the generated message (set `longMessage = False` on the class to replace it instead):

In [ ]:
class FailureMessages(unittest.TestCase):
    def test_numbers(self):
        self.assertEqual(apply_discount(80, 25), 65, msg="25% off 80")

    def test_lists(self):
        self.assertEqual(sorted(["b", "a", "c"]), ["a", "b", "d"])

run_quiet(FailureMessages)

## Fixtures: `setUp`, `tearDown` and friends

Like NUnit (and unlike xUnit, which creates a new class instance per test and uses the constructor), `unittest` calls `setUp()` before **each** test and `tearDown()` after it. Class-wide fixtures are **classmethods**.

```csharp
// NUnit
[OneTimeSetUp] public void OpenDatabase() { ... }
[SetUp]        public void CreateCart()   { cart = new Cart(); }
[TearDown]     public void ClearCart()    { ... }
```

`addCleanup(fn, *args)` registers teardown work as you go (like `using`/`IAsyncDisposable` stacking) — cleanups run in reverse order even if `setUp` fails half-way.

In [ ]:
class CartTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        print("setUpClass   (once)")
        cls.catalogue = {"apple": 0.5, "pear": 0.75}

    @classmethod
    def tearDownClass(cls):
        print("tearDownClass (once)")

    def setUp(self):
        print("  setUp")
        self.cart: list[str] = []
        self.addCleanup(print, "  cleanup (registered in setUp)")

    def tearDown(self):
        print("  tearDown")

    def test_empty(self):
        print("    test_empty")
        self.assertEqual(len(self.cart), 0)

    def test_add(self):
        print("    test_add")
        self.cart.append("apple")
        self.assertEqual(sum(self.catalogue[i] for i in self.cart), 0.5)

run_quiet(CartTests)

Tests run in **alphabetical order of method name** (`test_add` before `test_empty`), not in source order. Never let tests depend on each other's state — each test gets a fresh `setUp`.

## Data-driven tests with `subTest`

There is no `[Theory]` attribute. Instead, loop over your cases inside one test method and wrap each iteration in `with self.subTest(...)`. The keyword arguments label the case in the report, and — crucially — **a failing case does not stop the loop**, so you see every failing row:

```csharp
[Theory]
[InlineData(100, 10, 90)]
[InlineData(50, 50, 25)]
public void Discount(decimal price, int pct, decimal expected) =>
    Assert.Equal(expected, Pricing.ApplyDiscount(price, pct));
```

In [ ]:
class DiscountTable(unittest.TestCase):
    cases = [
        (100, 10, 90),
        (50, 50, 25),
        (19.99, 100, 0),
        (80, 25, 65),     # wrong on purpose (should be 60)
        (10, 5, 9),       # wrong on purpose (should be 9.5)
    ]

    def test_discounts(self):
        for price, pct, expected in self.cases:
            with self.subTest(price=price, pct=pct):
                self.assertEqual(apply_discount(price, pct), expected)

run_quiet(DiscountTable)

Note that `testsRun` is still 1 — sub-tests are reported individually but count as one test.

**Tip:** without `subTest`, the first failing row would end the test and hide the rest. If you need real per-case test methods (e.g. so each shows up separately in an IDE), pytest's `@pytest.mark.parametrize` is the usual answer.

## Skipping and expected failures

| C# | Python |
|---|---|
| `[Fact(Skip = "reason")]` / `[Ignore("reason")]` | `@unittest.skip("reason")` |
| conditional skip (`Assert.SkipWhen` in xUnit v3, `Assert.Ignore()` in NUnit) | `@unittest.skipIf(cond, "reason")`, `@unittest.skipUnless(cond, "reason")`, or `self.skipTest("reason")` at runtime |
| known bug that should fail | `@unittest.expectedFailure` |

An `expectedFailure` test that unexpectedly **passes** is reported as an "unexpected success", and the run is then *not* successful — so you notice when the bug gets fixed.

In [ ]:
class SkipDemo(unittest.TestCase):
    @unittest.skip("not implemented yet")
    def test_refunds(self):
        self.fail("never runs")

    @unittest.skipIf(sys.platform == "nonexistent-os", "only skipped on a made-up platform")
    def test_runs_everywhere(self):
        self.assertTrue(True)

    @unittest.skipUnless(sys.version_info >= (3, 14), "needs Python 3.14+")
    def test_modern_python(self):
        self.assertGreaterEqual(sys.version_info[:2], (3, 14))

    def test_runtime_skip(self):
        if "PRICING_DB" not in {}:          # imagine: os.environ
            self.skipTest("no database configured")

    @unittest.expectedFailure
    def test_known_rounding_bug(self):
        self.assertEqual(0.1 + 0.2, 0.3)

unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(loader.loadTestsFromTestCase(SkipDemo))
print("done")

## Test files, layout and the command line

Outside a notebook, tests live in files. The default discovery convention is **files named `test*.py`** in packages/directories under the start directory, containing `TestCase` subclasses with `test*` methods:

```text
my_project/
├── pyproject.toml
├── src/
│   └── shop/
│       ├── __init__.py
│       └── pricing.py
└── tests/
    ├── __init__.py
    └── test_pricing.py        # from shop.pricing import apply_discount
```

```python
# tests/test_pricing.py
import unittest
from shop.pricing import apply_discount

class ApplyDiscountTests(unittest.TestCase):
    def test_discount_is_applied(self):
        self.assertEqual(apply_discount(100, 10), 90)

if __name__ == "__main__":     # lets you run: python tests/test_pricing.py
    unittest.main()
```

Running them — the equivalents of `dotnet test` and its `--filter` option:

```bash
python -m unittest                                    # discover from the current dir
python -m unittest discover -s tests -p "test_*.py"   # explicit start dir and pattern
python -m unittest -v                                 # verbose (one line per test)
python -m unittest tests.test_pricing                 # one module
python -m unittest tests.test_pricing.ApplyDiscountTests.test_discount_is_applied
python -m unittest -k discount                        # only tests whose name matches
python -m unittest -f                                 # fail fast: stop at first failure
uv run python -m unittest                             # same, inside the uv project env
```

The exit code is non-zero when anything fails, so CI pipelines work as you would expect.

## A word about pytest

`unittest` is in the standard library, but the de-facto community standard is **pytest**, a third-party package (`uv add --dev pytest`). It is to `unittest` roughly what xUnit.net is to MSTest — the thing most open-source projects actually use. Its key differences:

- tests are plain **functions** using the plain **`assert`** statement; pytest rewrites the assert to show both sides on failure
- fixtures are injected by **parameter name** (`def test_x(tmp_path): ...`) — similar to constructor injection of xUnit fixtures
- `@pytest.mark.parametrize` gives you real `[Theory]`-style parametrised tests

```python
# test_pricing.py — pytest style (not runnable here: pytest is not in the stdlib)
import pytest
from shop.pricing import apply_discount

@pytest.mark.parametrize("price, pct, expected", [(100, 10, 90), (50, 50, 25)])
def test_discount(price, pct, expected):
    assert apply_discount(price, pct) == expected

def test_negative_percent_raises():
    with pytest.raises(ValueError, match="0-100"):
        apply_discount(100, -5)
```

**Tip:** pytest also runs existing `unittest.TestCase` classes unchanged, so knowing `unittest` is never wasted — and `unittest.mock` *(see Standard Library › Testing & Debugging › 002 · Mocking with unittest.mock)* is used with both.

⚠️ **Gotcha:** `async def test_...` methods on a plain `TestCase` are never awaited. Use `unittest.IsolatedAsyncioTestCase` (it gives each test its own event loop, plus `asyncSetUp`/`asyncTearDown`). It cannot be demonstrated here because it refuses to start inside Jupyter's already-running event loop — run it from a test file instead:

```python
class FetchTests(unittest.IsolatedAsyncioTestCase):
    async def test_fetch(self):
        self.assertEqual(await fetch_price("apple"), 0.5)
```

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Test class | `[TestFixture]` / `[TestClass]` / any class (xUnit) | `class XTests(unittest.TestCase)` |
| Test method | `[Fact]` / `[Test]` / `[TestMethod]` | `def test_something(self)` |
| Equality | `Assert.Equal(expected, actual)` | `self.assertEqual(a, b)` |
| Float equality | `Assert.Equal(x, y, precision)` | `self.assertAlmostEqual(a, b)` |
| Null check | `Assert.Null(x)` | `self.assertIsNone(x)` |
| Expect exception | `Assert.Throws<T>(...)` | `with self.assertRaises(T) as cm:` |
| Per-test setup | `[SetUp]` / constructor | `setUp()` / `tearDown()` / `addCleanup()` |
| Per-class setup | `[OneTimeSetUp]` / `IClassFixture<T>` | `@classmethod setUpClass` / `tearDownClass` |
| Data-driven | `[Theory]` + `[InlineData]` | loop + `with self.subTest(...)` |
| Skip | `[Fact(Skip=...)]` / `[Ignore]` | `@unittest.skip`, `skipIf`, `skipUnless`, `self.skipTest()` |
| Run in notebook | — | `TextTestRunner(stream=sys.stdout).run(loader.loadTestsFromTestCase(X))` |
| Run from CLI | `dotnet test` | `python -m unittest [discover] [-v] [-k pattern]` |

## Exercises

1. Write a function `is_palindrome(s: str) -> bool` that ignores case and spaces, and a `TestCase` with at least three tests (a palindrome, a non-palindrome, and the empty string). Run it with `TextTestRunner`.
2. Write a `StackTests` class for a plain Python `list` used as a stack. Use `setUp` to create a stack containing `[1, 2, 3]`, and test that `pop()` returns `3` and that popping an empty list raises `IndexError` (check the message contains `"empty"` with `assertRaisesRegex`).
3. Convert this `[Theory]` into a single `unittest` test using `subTest`: `[InlineData("10", 10)] [InlineData(" 7 ", 7)] [InlineData("-3", -3)]` testing `int(text) == expected`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import sys
import unittest

# 1
def is_palindrome(s: str) -> bool:
    cleaned = s.replace(" ", "").lower()
    return cleaned == cleaned[::-1]

class PalindromeTests(unittest.TestCase):
    def test_palindrome(self):
        self.assertTrue(is_palindrome("Never odd or even"))

    def test_not_palindrome(self):
        self.assertFalse(is_palindrome("python"))

    def test_empty_string(self):
        self.assertTrue(is_palindrome(""))

# 2
class StackTests(unittest.TestCase):
    def setUp(self):
        self.stack = [1, 2, 3]

    def test_pop_returns_last(self):
        self.assertEqual(self.stack.pop(), 3)

    def test_pop_empty_raises(self):
        self.stack.clear()
        with self.assertRaisesRegex(IndexError, "empty"):
            self.stack.pop()

# 3
class ParseIntTests(unittest.TestCase):
    def test_parse(self):
        for text, expected in [("10", 10), (" 7 ", 7), ("-3", -3)]:
            with self.subTest(text=text):
                self.assertEqual(int(text), expected)

loader = unittest.TestLoader()
suite = unittest.TestSuite(
    loader.loadTestsFromTestCase(c) for c in (PalindromeTests, StackTests, ParseIntTests)
)
unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
```

</details>